# 3. Feature Engineering

- Load wrangled dataset
- Create cyclical date features (sin/cos encoding)
- Create bidder-count target classes (`TARGET_CLASS`)
- Create logarithmic value categories
- Drop original date, bidder count, and value columns
- Split 80/20 into train and test sets
- Drop identifier and constant columns
- Impute missing values separately for train and test
- Target-encode categorical columns (fit on train, transform both)
- Feature selection using Mutual Information (fit on train only)
- Save train and test sets as separate files

In [36]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, mutual_info_classif

ROOT = Path(globals()["__vsc_ipynb_file__"]).parent.parent
INPUT_PATH = ROOT / "data" / "interim" / "ted_awards_wrangled.csv"
TRAIN_PATH = ROOT / "data" / "processed" / "train.csv"
TEST_PATH = ROOT / "data" / "processed" / "test.csv"

df = pd.read_csv(INPUT_PATH)
print("Loaded shape:", df.shape)
df.head()

Loaded shape: (1830, 44)


,ID_NOTICE_CAN,TED_NOTICE_URL,YEAR,ID_TYPE,DT_DISPATCH,XSD_VERSION,CANCELLED,CORRECTIONS,B_MULTIPLE_CAE,CAE_NAME,...,VALUE_EURO,VALUE_EURO_FIN_1,VALUE_EURO_FIN_2,B_EU_FUNDS,TOP_TYPE,OUT_OF_DIRECTIVES,CRIT_CODE,B_ELECTRONIC_AUCTION,NUMBER_AWARDS,NUMBER_OFFERS
0,20184372,ted.europa.eu/udl?uri=TED:NOTICE:4372-2018:TEX...,2018,3,43103,R209.S2,0,0,N,SYKEHUSINNKJØP HF,...,33342016.15,8648085.43,8648085.43,N,OPE,0,M,N,6,7.0
1,20184372,ted.europa.eu/udl?uri=TED:NOTICE:4372-2018:TEX...,2018,3,43103,R209.S2,0,0,N,SYKEHUSINNKJØP HF,...,33342016.15,8648085.43,8648085.43,N,OPE,0,M,N,6,4.0
2,20184372,ted.europa.eu/udl?uri=TED:NOTICE:4372-2018:TEX...,2018,3,43103,R209.S2,0,0,N,SYKEHUSINNKJØP HF,...,33342016.15,8648085.43,8648085.43,N,OPE,0,M,N,6,7.0
3,20184372,ted.europa.eu/udl?uri=TED:NOTICE:4372-2018:TEX...,2018,3,43103,R209.S2,0,0,N,SYKEHUSINNKJØP HF,...,33342016.15,8648085.43,8648085.43,N,OPE,0,M,N,6,4.0
4,20184372,ted.europa.eu/udl?uri=TED:NOTICE:4372-2018:TEX...,2018,3,43103,R209.S2,0,0,N,SYKEHUSINNKJØP HF,...,33342016.15,8648085.43,8648085.43,N,OPE,0,M,N,6,6.0


In [37]:
## Cyclical Date Encoding (Sin/Cos)

#ses are cyclical — December 31 is close to January 1, but a raw day-of-year number (1 vs 365) doesn't capture that.

#e encode the **day of year** and **month** as pairs of sine and cosine values:

#$\text{sin\_feature} = \sin\!\left(\frac{2\pi \cdot x}{\text{period}}\right), \quad \text{cos\_feature} = \cos\!\left(\frac{2\pi \cdot x}{\text{period}}\right)$$

#here $x$ is the day of year (period = 365) or month (period = 12). This maps the cycle onto a circle so that the model sees similar values for nearby dates, even across the year boundary.

In [38]:
# Convert Excel serial date to datetime
df["DATE"] = pd.to_datetime(df["DT_DISPATCH"], origin="1899-12-30", unit="D", errors="coerce")

# Extract day of year and month
day_of_year = df["DATE"].dt.dayofyear
month = df["DATE"].dt.month

# Cyclical sin/cos encoding
df["DAY_SIN"] = np.sin(2 * np.pi * day_of_year / 365)
df["DAY_COS"] = np.cos(2 * np.pi * day_of_year / 365)
df["MONTH_SIN"] = np.sin(2 * np.pi * month / 12)
df["MONTH_COS"] = np.cos(2 * np.pi * month / 12)

print("Date features created:")
df[["DT_DISPATCH", "DATE", "DAY_SIN", "DAY_COS", "MONTH_SIN", "MONTH_COS"]].head()

Date features created:


,DT_DISPATCH,DATE,DAY_SIN,DAY_COS,MONTH_SIN,MONTH_COS
0,43103,2018-01-03,0.05162,0.998667,0.5,0.866025
1,43103,2018-01-03,0.05162,0.998667,0.5,0.866025
2,43103,2018-01-03,0.05162,0.998667,0.5,0.866025
3,43103,2018-01-03,0.05162,0.998667,0.5,0.866025
4,43103,2018-01-03,0.05162,0.998667,0.5,0.866025


In [39]:
# Create TARGET_CLASS from NUMBER_OFFERS: 1, 2-3, 4-6, 7+
def classify_offers(n):
    if pd.isna(n): return np.nan
    n = int(n)
    if n == 1: return 1
    if 2 <= n <= 3: return 2
    if 4 <= n <= 6: return 3
    if n >= 7: return 4
    return np.nan

df["TARGET_CLASS"] = df["NUMBER_OFFERS"].apply(classify_offers)

# Create logarithmic VALUE_EURO categories: Small, Medium, Large
df["VALUE_EURO_LOG"] = np.log1p(pd.to_numeric(df["VALUE_EURO"], errors="coerce"))
thresholds = df["VALUE_EURO_LOG"].quantile([1/3, 2/3])
df["VALUE_CATEGORY"] = pd.cut(
    df["VALUE_EURO_LOG"],
    bins=[-np.inf, thresholds.iloc[0], thresholds.iloc[1], np.inf],
    labels=["Small", "Medium", "Large"],
)

print("TARGET_CLASS distribution:")
print(df["TARGET_CLASS"].value_counts(dropna=False).sort_index())
print("\nVALUE_CATEGORY distribution:")
print(df["VALUE_CATEGORY"].value_counts(dropna=False))

TARGET_CLASS distribution:
TARGET_CLASS
1    174
2    737
3    678
4    241
Name: count, dtype: int64

VALUE_CATEGORY distribution:
VALUE_CATEGORY
Small     590
Medium    589
Large     589
NaN        62
Name: count, dtype: int64


In [40]:
# Drop original date, bidder count, and value columns
df = df.drop(columns=["DT_DISPATCH", "DATE", "NUMBER_OFFERS", "VALUE_EURO"])

# Drop rows without a valid target
df = df.dropna(subset=["TARGET_CLASS"]).reset_index(drop=True)
df["TARGET_CLASS"] = df["TARGET_CLASS"].astype(int)

print("Shape after dropping originals and rows without target:", df.shape)
print("Columns:", list(df.columns))

Shape after dropping originals and rows without target: (1830, 48)
Columns: ['ID_NOTICE_CAN', 'TED_NOTICE_URL', 'YEAR', 'ID_TYPE', 'XSD_VERSION', 'CANCELLED', 'CORRECTIONS', 'B_MULTIPLE_CAE', 'CAE_NAME', 'CAE_NATIONALID', 'CAE_ADDRESS', 'CAE_TOWN', 'CAE_POSTAL_CODE', 'CAE_GPA_ANNEX', 'ISO_COUNTRY_CODE', 'ISO_COUNTRY_CODE_GPA', 'B_MULTIPLE_COUNTRY', 'CAE_TYPE', 'MAIN_ACTIVITY', 'B_ON_BEHALF', 'B_INVOLVES_JOINT_PROCUREMENT', 'B_AWARDED_BY_CENTRAL_BODY', 'TYPE_OF_CONTRACT', 'TAL_LOCATION_NUTS', 'B_FRA_AGREEMENT', 'B_FRA_CONTRACT', 'B_DYN_PURCH_SYST', 'CPV', 'MAIN_CPV_CODE_GPA', 'ADDITIONAL_CPVS', 'B_GPA', 'GPA_COVERAGE', 'LOTS_NUMBER', 'VALUE_EURO_FIN_1', 'VALUE_EURO_FIN_2', 'B_EU_FUNDS', 'TOP_TYPE', 'OUT_OF_DIRECTIVES', 'CRIT_CODE', 'B_ELECTRONIC_AUCTION', 'NUMBER_AWARDS', 'DAY_SIN', 'DAY_COS', 'MONTH_SIN', 'MONTH_COS', 'TARGET_CLASS', 'VALUE_EURO_LOG', 'VALUE_CATEGORY']


In [41]:
# 80/20 train/test split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["TARGET_CLASS"])
train_df = train_df.copy()
test_df = test_df.copy()
print(f"Train: {train_df.shape}, Test: {test_df.shape}")

Train: (1464, 48), Test: (366, 48)


In [42]:
# Drop true identifier and constant columns (unique per row or zero variance)
ID_COLS = ["TED_NOTICE_URL"]
CONSTANT_COLS = ["CAE_GPA_ANNEX", "ISO_COUNTRY_CODE", "ISO_COUNTRY_CODE_GPA", "B_MULTIPLE_COUNTRY"]
drop_cols = [c for c in ID_COLS + CONSTANT_COLS if c in train_df.columns]
train_df = train_df.drop(columns=drop_cols)
test_df = test_df.drop(columns=drop_cols)
print(f"Dropped {len(drop_cols)} identifier/constant columns: {drop_cols}")

# Identify column types
numeric_cols = train_df.select_dtypes(include=["number"]).columns.drop("TARGET_CLASS").tolist()
categorical_cols = train_df.select_dtypes(exclude=["number"]).columns.tolist()
print(f"Numeric: {len(numeric_cols)}, Categorical: {len(categorical_cols)}")
print(f"Categorical columns: {categorical_cols}")

# Impute numeric (train in isolation, test in isolation)
for col in numeric_cols:
    train_df[col] = train_df[col].fillna(train_df[col].median())
    train_df[col] = train_df[col].fillna(0)
for col in numeric_cols:
    test_df[col] = test_df[col].fillna(test_df[col].median())
    test_df[col] = test_df[col].fillna(0)

# Impute categorical (fill NaN before encoding)
for col in categorical_cols:
    mode_val = train_df[col].mode()
    train_df[col] = train_df[col].fillna(mode_val.iloc[0] if not mode_val.empty else "MISSING")
for col in categorical_cols:
    mode_val = test_df[col].mode()
    test_df[col] = test_df[col].fillna(mode_val.iloc[0] if not mode_val.empty else "MISSING")

print(f"Train NaNs after imputation: {train_df.isna().sum().sum()}")
print(f"Test NaNs after imputation: {test_df.isna().sum().sum()}")

# Label-encode categorical columns (fit on train, transform both; unseen values get -1)
from sklearn.preprocessing import LabelEncoder

label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    le.fit(train_df[col].astype(str))
    train_df[col] = le.transform(train_df[col].astype(str))
    test_df[col] = test_df[col].astype(str).map(
        lambda v, _le=le: _le.transform([v])[0] if v in _le.classes_ else -1
    )
    label_encoders[col] = le

print(f"\nAfter label encoding:")
print(f"Train: {train_df.shape}, Test: {test_df.shape}")

Dropped 5 identifier/constant columns: ['TED_NOTICE_URL', 'CAE_GPA_ANNEX', 'ISO_COUNTRY_CODE', 'ISO_COUNTRY_CODE_GPA', 'B_MULTIPLE_COUNTRY']
Numeric: 18, Categorical: 24
Categorical columns: ['XSD_VERSION', 'B_MULTIPLE_CAE', 'CAE_NAME', 'CAE_NATIONALID', 'CAE_ADDRESS', 'CAE_TOWN', 'CAE_POSTAL_CODE', 'CAE_TYPE', 'MAIN_ACTIVITY', 'B_ON_BEHALF', 'B_INVOLVES_JOINT_PROCUREMENT', 'B_AWARDED_BY_CENTRAL_BODY', 'TYPE_OF_CONTRACT', 'TAL_LOCATION_NUTS', 'B_FRA_AGREEMENT', 'B_FRA_CONTRACT', 'B_DYN_PURCH_SYST', 'ADDITIONAL_CPVS', 'B_GPA', 'B_EU_FUNDS', 'TOP_TYPE', 'CRIT_CODE', 'B_ELECTRONIC_AUCTION', 'VALUE_CATEGORY']
Train NaNs after imputation: 0
Test NaNs after imputation: 0

After label encoding:
Train: (1464, 43), Test: (366, 43)


In [43]:
# Feature selection: Mutual Information + correlation filter
X_train = train_df.drop(columns=["TARGET_CLASS"])
y_train = train_df["TARGET_CLASS"]

# Step 1: Score all features with Mutual Information
selector = SelectKBest(score_func=mutual_info_classif, k="all")
selector.fit(X_train, y_train)

scores = pd.DataFrame({
    "feature": X_train.columns,
    "mi_score": selector.scores_,
    "corr_abs": X_train.corrwith(y_train).abs().values,
}).sort_values("mi_score", ascending=False)

print("All feature scores:")
print(scores.to_string(index=False))

# Step 2: Keep features that have decent MI AND are not pure noise
# Require MI >= 0.02 AND correlation >= 0.01
selected_cols = scores[
    (scores["mi_score"] >= 0.02) & (scores["corr_abs"] >= 0.01)
]["feature"].tolist()

dropped = [f for f in X_train.columns if f not in selected_cols]
print(f"\nDropped {len(dropped)} low-signal features: {dropped}")
print(f"Keeping {len(selected_cols)} features")

# Keep only selected features + target
train_df = train_df[selected_cols + ["TARGET_CLASS"]]
test_df = test_df[selected_cols + ["TARGET_CLASS"]]
print(f"Train: {train_df.shape}, Test: {test_df.shape}")

All feature scores:
                     feature  mi_score  corr_abs
             CAE_POSTAL_CODE  0.082991  0.071417
             ADDITIONAL_CPVS  0.081382  0.055623
              VALUE_EURO_LOG  0.061545  0.056993
                 CAE_ADDRESS  0.055738  0.015481
                     DAY_SIN  0.050963  0.002289
            VALUE_EURO_FIN_1  0.049478  0.053259
                    CAE_NAME  0.046444  0.036999
            VALUE_EURO_FIN_2  0.046111  0.054127
              CAE_NATIONALID  0.045223  0.001001
                     DAY_COS  0.044652  0.056432
               ID_NOTICE_CAN  0.043020  0.014955
                    CAE_TYPE  0.040581  0.048405
           TAL_LOCATION_NUTS  0.037950  0.028233
              VALUE_CATEGORY  0.034669  0.126791
                 LOTS_NUMBER  0.030253  0.010237
            B_DYN_PURCH_SYST  0.028120  0.001719
                     ID_TYPE  0.027003  0.019366
B_INVOLVES_JOINT_PROCUREMENT  0.026300  0.051192
                   CANCELLED  0.025524       NaN


c:\Users\Johannes Espeset\Downloads\Master-Class-main\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\Johannes Espeset\Downloads\Master-Class-main\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [44]:
# Save train and test sets
TRAIN_PATH.parent.mkdir(parents=True, exist_ok=True)
train_df.to_csv(TRAIN_PATH, index=False)
test_df.to_csv(TEST_PATH, index=False)

print(f"Saved train set: {TRAIN_PATH} ({train_df.shape})")
print(f"Saved test set: {TEST_PATH} ({test_df.shape})")

Saved train set: c:\Users\Johannes Espeset\Downloads\Master-Class-main\Master-Class-main\data\processed\train.csv ((1464, 18))
Saved test set: c:\Users\Johannes Espeset\Downloads\Master-Class-main\Master-Class-main\data\processed\test.csv ((366, 18))
